In [1]:


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_DIR = "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge"

# ============================================================
# LOAD
# ============================================================
train = pd.read_csv(f"{DATA_DIR}/train.csv", low_memory=False)
print("Train shape:", train.shape)

# ============================================================
# SECTION 1 — FEATURE ENGINEERING (on the Training Data)
# ============================================================
train['TransactionDate'] = pd.to_datetime(train['TransactionDate'])

train['TransactionYear']    = train['TransactionDate'].dt.year
train['TransactionMonth']   = train['TransactionDate'].dt.month
train['TransactionQuarter'] = train['TransactionDate'].dt.quarter
train['ManufactureDecade']  = (train['ManufactureYear'] // 10) * 10
train['SpecificationComplexity'] = (
    train['Spec_FullDescriptor'].astype(str).str.split().str.len()
)

# --- Q1: which ManufactureDecade has the highest average TargetValue? ---
decade_avg = train.groupby('ManufactureDecade')['TargetValue'].mean().sort_values(ascending=False)
print("\n[Q1] Average TargetValue by ManufactureDecade:")
print(decade_avg)
print("[Q1] ANSWER (highest-avg decade):", decade_avg.index[0])

# --- Q2: maximum value of SpecificationComplexity ---
q2_answer = train['SpecificationComplexity'].max()
print("\n[Q2] ANSWER (max SpecificationComplexity):", round(float(q2_answer), 2))

# ============================================================
# SECTION 2 — FEATURE SELECTION (build the new Training Data)
# ============================================================
categorical_features = [
    'DataOriginCode', 'VendorPartnerID', 'UtilizationTier', 'AssetScaleFactor',
    'RegionCode', 'InventoryGroupCategory', 'InventoryGroupDescription',
]

# "all numerical features except TransactionID, AssetID"
# -> every numeric-dtype column in the (feature-engineered) train frame,
#    excluding TransactionID, AssetID, and the target TargetValue itself.
numeric_dtype_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numerical_features = [
    c for c in numeric_dtype_cols
    if c not in ['TransactionID', 'AssetID', 'TargetValue']
]

feature_cols = categorical_features + numerical_features
print("\nCategorical features:", categorical_features)
print("Numerical features:", numerical_features)

X = train[feature_cols].copy()
y = train['TargetValue'].copy()

# --- Q3: number of features in the (new, pre-preprocessing) Training Data ---
print("\n[Q3] ANSWER (number of features):", X.shape[1])

# ============================================================
# SECTION 3 — TRAIN/VALIDATION SPLIT (before any preprocessing)
# ============================================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("\nX_train shape:", X_train.shape, "| X_val shape:", X_val.shape)

# ============================================================
# SECTION 4 — PREPROCESSING (Pipeline + ColumnTransformer)
# ============================================================
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features),
])

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

# handle sparse output from OneHotEncoder
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()
    X_val_processed = X_val_processed.toarray()

feature_names = preprocessor.get_feature_names_out()

# --- Q4: shape of the resultant training dataset after preprocessing ---
print("\n[Q4] ANSWER (processed training shape):", X_train_processed.shape)

# ============================================================
# SECTION 5 — FEATURE SELECTION (SelectKBest, f_regression, top 100)
# ============================================================
K = 100
selector = SelectKBest(score_func=f_regression, k=K)
X_train_selected = selector.fit_transform(X_train_processed, y_train)
X_val_selected = selector.transform(X_val_processed)

scores = selector.scores_
support_mask = selector.get_support()

scores_df = pd.DataFrame({'feature': feature_names, 'f_score': scores})
selected_df = scores_df[support_mask].sort_values('f_score', ascending=False).reset_index(drop=True)
removed_df  = scores_df[~support_mask].sort_values('f_score', ascending=False).reset_index(drop=True)

print("\nTop 10 selected features by F-score:")
print(selected_df.head(10).to_string(index=False))

# --- Q5: highest F-score feature (among selected/top-100) ---
print("\n[Q5] ANSWER (highest F-score feature):", selected_df.iloc[0]['feature'])

# --- Q6: lowest F-score feature among the SELECTED (top-100) features ---
print("[Q6] ANSWER (lowest F-score among selected):", selected_df.iloc[-1]['feature'])

# --- Q7: which features were REMOVED by SelectKBest (not in the top 100) ---
print("\nAll removed features (not in top 100), sorted by F-score:")
print(removed_df.to_string(index=False))
print("[Q7] Check which of the given options appear in the 'removed_df' list above.")

# ============================================================
# SECTION 6 — BASELINE MODEL (RandomForestRegressor, default params)
# ============================================================
rf_baseline = RandomForestRegressor(random_state=42)
rf_baseline.fit(X_train_selected, y_train)

val_pred_baseline = rf_baseline.predict(X_val_selected)

baseline_mae = mean_absolute_error(y_val, val_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_val, val_pred_baseline))

print("\n[Q8] ANSWER (baseline validation MAE):", round(baseline_mae, 2))
print("[Q9] ANSWER (baseline validation RMSE):", round(baseline_rmse, 2))

# ============================================================
# SECTION 7 — HYPERPARAMETER TUNING (RandomizedSearchCV)
# ============================================================
param_distributions = {
    'n_estimators': [100, 200],
    'max_depth': [10, None],
    'max_features': ['sqrt'],
}

rf_search = RandomForestRegressor(random_state=42)

random_search = RandomizedSearchCV(
    estimator=rf_search,
    param_distributions=param_distributions,
    n_iter=4,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
)
random_search.fit(X_train_selected, y_train)

best_params = random_search.best_params_
best_cv_rmse = -random_search.best_score_  # scorer is negative RMSE

print("\nBest params found:", best_params)
print("[Q10] ANSWER (best n_estimators):", best_params['n_estimators'])
print("[Q11] ANSWER (best max_depth):", best_params['max_depth'])
print("[Q12] ANSWER (best CV RMSE):", round(best_cv_rmse, 2))

# --- Q13/Q14: validation MAE/RMSE of the tuned model ---
best_rf = random_search.best_estimator_
val_pred_tuned = best_rf.predict(X_val_selected)

tuned_mae = mean_absolute_error(y_val, val_pred_tuned)
tuned_rmse = np.sqrt(mean_squared_error(y_val, val_pred_tuned))

print("\n[Q13] ANSWER (tuned validation MAE):", round(tuned_mae, 2))
print("[Q14] ANSWER (tuned validation RMSE):", round(tuned_rmse, 2))

print("\n" + "=" * 60)
print("ALL ANSWERS PRINTED ABOVE — scroll up and copy each [Qn] value.")
print("=" * 60)

Train shape: (138701, 50)

[Q1] Average TargetValue by ManufactureDecade:
ManufactureDecade
2010    59560.064935
2000    48659.923834
1990    41559.931040
1940    37500.000000
1980    35982.492087
1920    28925.000000
1000    26428.737346
1970    25893.403433
1960    12606.947162
1950     9703.472222
Name: TargetValue, dtype: float64
[Q1] ANSWER (highest-avg decade): 2010

[Q2] ANSWER (max SpecificationComplexity): 4.0

Categorical features: ['DataOriginCode', 'VendorPartnerID', 'UtilizationTier', 'AssetScaleFactor', 'RegionCode', 'InventoryGroupCategory', 'InventoryGroupDescription']
Numerical features: ['ProductConfigID', 'ManufactureYear', 'OperationalHoursMeter', 'TransactionYear', 'TransactionMonth', 'TransactionQuarter', 'ManufactureDecade', 'SpecificationComplexity']

[Q3] ANSWER (number of features): 15

X_train shape: (110960, 15) | X_val shape: (27741, 15)

[Q4] ANSWER (processed training shape): (110960, 118)

Top 10 selected features by F-score:
                            